In [ ]:
import concurrent.futures
import os
import re
from google.cloud import storage
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ==========================================
# CONFIGURATION
# ==========================================
GCS_PATH_RUN_A = "gs://fc-secure-8c7c6bb6-6241-4a03-b87d-895b5abfd91d/submissions/intermediates/60866988-3ed6-41ea-849f-a1f176f54211/GLIMPSE2Concordance/48127ec8-c787-4935-bf8f-ef4a83569623"

# This is the baseline 198 sample eval we should be comparing against. You shouldnt have to change this
#GCS_PATH_RUN_B = "gs://fc-secure-e9018a40-98de-4d5d-8f40-16e60c8f8a0b/submissions/e3080e69-a99a-48ff-8513-bdfc926ce614/GLIMPSE2Concordance/50757920-0810-47c2-b594-8b449d36c050"

# This is the baseline 284 sample eval we should be comparing against. You shouldnt have to change this
GCS_PATH_RUN_B = "gs://fc-secure-e9018a40-98de-4d5d-8f40-16e60c8f8a0b/submissions/efcab7cb-40ac-44d6-84e1-8594ac0c3306/GLIMPSE2Concordance/748d8161-6c4f-4f20-9673-7932f14521d7"

LABEL_RUN_A = "Replication"
LABEL_RUN_B = "Baseline"

OUTPUT_PREFIX = "concordance_difference"


# ==========================================
# HELPER FUNCTIONS
# ==========================================
def parse_gcs_url(gcs_url: str):
    match = re.match(r"gs://([^/]+)/(.*)", gcs_url)
    if not match:
        raise ValueError(f"Invalid GCS URL: {gcs_url}")
    bucket_name, prefix = match.groups()
    return bucket_name, prefix.rstrip("/")


def find_rsquare_files_gcs(gcs_url: str, storage_client: storage.Client):
    """Find all .rsquare.grp.txt.gz files, filtering out older preempted attempts."""
    bucket_name, prefix = parse_gcs_url(gcs_url)
    bucket = storage_client.bucket(bucket_name)

    blobs = bucket.list_blobs(prefix=prefix)
    
    latest_files = {}
    total_rsquare_seen = 0

    for blob in blobs:
        if not blob.name.endswith(".rsquare.grp.txt.gz"):
            continue
            
        total_rsquare_seen += 1
        filename = os.path.basename(blob.name)
        full_path = f"gs://{bucket_name}/{blob.name}"
        
        attempt_match = re.search(r"/attempt-(\d+)/", blob.name)
        attempt_num = int(attempt_match.group(1)) if attempt_match else 0
        
        if filename not in latest_files or attempt_num > latest_files[filename]["attempt"]:
            latest_files[filename] = {
                "attempt": attempt_num,
                "path": full_path
            }

    rsquare_files = [file_info["path"] for file_info in latest_files.values()]

    print(f"Found {len(rsquare_files)} unique rsquare files in {gcs_url} "
          f"(filtered {total_rsquare_seen - len(rsquare_files)} preempted/older files)")
    return rsquare_files


def parse_rsquare_filepath(filepath: str):
    filename = os.path.basename(filepath)
    parts = filename.split("_GPfilt_")
    prefix_parts = parts[0].split(".")
    is_info05 = "INFO05" in prefix_parts

    # WDL output naming convention relies on suffix additions:
    # prefix.region.trh_bin.length_bin.[INFO05.]concordance-result
    if is_info05:
        length_bin = prefix_parts[-3]
        trh_bin = prefix_parts[-4]
        region = prefix_parts[-5]
        label_type = "INFO05"
    else:
        length_bin = prefix_parts[-2]
        trh_bin = prefix_parts[-3]
        region = prefix_parts[-4]
        label_type = "NORMAL"

    min_tar_gp = float(parts[1].split(".rsquare")[0])
    return region, trh_bin, length_bin, min_tar_gp, label_type


def read_single_rsquare_file(filepath: str):
    region, trh_bin, length_bin, min_tar_gp, label_type = parse_rsquare_filepath(filepath)
    df = pd.read_csv(
        filepath,
        sep=r"\s+",
        comment="#",
        names=[
            "AF_BIN_INDEX",
            "AF_BIN_COUNT",
            "AF_BIN_MEAN",
            "R2_GT",
            "R2_DS",
        ],
        compression="gzip",
    )
    
    df = df[df["AF_BIN_COUNT"] > 0].copy()
    
    df["REGION"] = region
    df["TRH_BIN"] = trh_bin
    df["LENGTH_BIN"] = length_bin
    df["MIN_TAR_GP"] = min_tar_gp
    df["LABEL_TYPE"] = label_type
    
    return df


def load_run_data(gcs_files: list, max_workers: int = 16):
    """Loads all data and aggregates it at the Region level."""
    dataframes = []
    print(f"  Downloading and reading {len(gcs_files)} files using {max_workers} threads...")
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_url = {executor.submit(read_single_rsquare_file, url): url for url in gcs_files}
        for future in concurrent.futures.as_completed(future_to_url):
            url = future_to_url[future]
            try:
                df = future.result()
                if not df.empty:
                    dataframes.append(df)
            except Exception as exc:
                print(f"  [Error] {url} generated an exception: {exc}")

    if not dataframes:
        return pd.DataFrame()

    df_all = pd.concat(dataframes, ignore_index=True)
    df_all["WEIGHTED_R2"] = df_all["R2_DS"] * df_all["AF_BIN_COUNT"]
    df_all["WEIGHTED_AF"] = df_all["AF_BIN_MEAN"] * df_all["AF_BIN_COUNT"]

    # Aggregate at the region level (handles potential sharding within regions gracefully)
    df_region = (
        df_all.groupby([
            "REGION",
            "TRH_BIN",
            "LENGTH_BIN",
            "MIN_TAR_GP",
            "LABEL_TYPE",
            "AF_BIN_INDEX",
        ])
        .agg(
            {
                "AF_BIN_COUNT": "sum",
                "WEIGHTED_R2": "sum",
                "WEIGHTED_AF": "sum",
            }
        )
        .reset_index()
    )

    df_region["R2_DS"] = df_region["WEIGHTED_R2"] / df_region["AF_BIN_COUNT"]
    df_region["AF_BIN_MEAN"] = df_region["WEIGHTED_AF"] / df_region["AF_BIN_COUNT"]

    return df_region


def aggregate_all_regions(df_region: pd.DataFrame):
    """Aggregates region-level data across all chromosomes into a single aggregate dataset."""
    df_agg = (
        df_region.groupby([
            "TRH_BIN",
            "LENGTH_BIN",
            "MIN_TAR_GP",
            "LABEL_TYPE",
            "AF_BIN_INDEX",
        ])
        .agg(
            {
                "AF_BIN_COUNT": "sum",
                "WEIGHTED_R2": "sum",
                "WEIGHTED_AF": "sum",
            }
        )
        .reset_index()
    )

    df_agg["R2_DS"] = df_agg["WEIGHTED_R2"] / df_agg["AF_BIN_COUNT"]
    df_agg["AF_BIN_MEAN"] = df_agg["WEIGHTED_AF"] / df_agg["AF_BIN_COUNT"]
    
    return df_agg


# ==========================================
# PLOTTING FUNCTION
# ==========================================
def plot_differences(merged_df: pd.DataFrame, title_metadata: str, output_prefix: str):
    """Generates the multi-panel difference plots for a given merged DataFrame."""
    if merged_df.empty:
        return

    for trh_bin in ["outTRH", "inTRH"]:
        fig, ax = plt.subplots(1, 5, figsize=(12, 2.5), sharey=True)

        for i, length_bin in enumerate(["SV_DEL", "DEL", "SNP", "INS", "SV_INS"]):
            ax2 = ax[i].twinx()
            
            # Apply uniform [-1, 1] y-limits for both axes to perfectly align 0
            ax[i].set_ylim([-1, 1])
            ax2.set_ylim([-1, 1])

            for label_type, min_tar_gp in [("NORMAL", 0.0), ("NORMAL", 0.9), ("INFO05", 0.0)]:
                if label_type == "INFO05":
                    label = "INFO > 0.5"
                    ls_val = "dashed"
                else:
                    label = "unfiltered" if min_tar_gp == 0.0 else f"GP > {min_tar_gp}"
                    ls_val = "solid" if min_tar_gp == 0.0 else "dotted"

                mask = (
                    (merged_df["TRH_BIN"] == trh_bin)
                    & (merged_df["LENGTH_BIN"] == length_bin)
                    & (merged_df["MIN_TAR_GP"] == min_tar_gp)
                    & (merged_df["LABEL_TYPE"] == label_type)
                )

                sub_df = merged_df[mask].sort_values("AF_BIN_MEAN")

                if not sub_df.empty:
                    # Primary Axis: Absolute R2 Difference
                    ax[i].plot(
                        sub_df["AF_BIN_MEAN"],
                        sub_df["R2_ABS_DIFF"],
                        label=label,
                        ls=ls_val,
                        color="C0",
                    )

                    # Secondary Axis: Relative Count Difference
                    ax2.plot(
                        sub_df["AF_BIN_MEAN"],
                        sub_df["COUNT_REL_DIFF"],
                        label=f"{label} (count diff)",
                        ls=ls_val,
                        color="C1",
                        alpha=0.5,
                    )

            # Baseline at 0 for primary axis
            ax[i].axhline(0, color="black", linestyle="--", linewidth=0.8)

            ax[i].set_xscale("log")
            ax[i].set_xlim([1e-4, 1])

            # Format axis labels and ticks
            if i == 0:
                ax[i].set_ylabel(
                    r"$r^2_{A} - r^2_{B}$",
                    fontsize=14,
                    color="C0",
                )
                ax2.set_yticklabels([])
            elif i == 4:
                ax2.set_ylabel(
                    r"$\frac{n_{A} - n_{B}}{n_{B}}$ variants",
                    fontsize=14,
                    color="C1",
                    rotation=270,
                    va="bottom",
                )
            else:
                ax2.set_yticklabels([])

            # Panel headers and footers
            if i == 2:
                trh_tag = {"outTRH": "non-TR/homopolymer", "inTRH": "TR/homopolymer"}[trh_bin]
                ax[i].set_title(f"{title_metadata}\n{trh_tag}\n", fontsize=10)
                ax[i].set_xlabel(
                    f"panel allele frequency\n\n{length_bin}\nALT length - REF length (bp)",
                    fontsize=11,
                )
                ax[i].legend(loc="lower left", fontsize=7)
            else:
                length_bin_label = {
                    "SV_DEL": "(-inf, -50]",
                    "DEL": "(-50, -1]",
                    "SNP": "SNP",
                    "INS": "[0, 50)",
                    "SV_INS": "[50, inf)",
                }[length_bin]
                ax[i].set_xlabel(f"\n\n{length_bin_label}", fontsize=11)

        plt.show()
        
        plt.savefig(f"{output_prefix}.{trh_bin}.diff.png")
        plt.savefig(f"{output_prefix}.{trh_bin}.diff.pdf")
        
        print(f"Saved absolute difference plots for {trh_bin} to {output_prefix}.{trh_bin}.diff.[png|pdf]")
        plt.close()


# ==========================================
# MAIN EXECUTION
# ==========================================
def main():
    storage_client = storage.Client()

    files_a = find_rsquare_files_gcs(GCS_PATH_RUN_A, storage_client)
    files_b = find_rsquare_files_gcs(GCS_PATH_RUN_B, storage_client)

    if not files_a or not files_b:
        raise ValueError("Could not find .rsquare.grp.txt.gz files in one or both directories.")

    print("Processing Run A...")
    df_region_a = load_run_data(files_a)

    print("Processing Run B...")
    df_region_b = load_run_data(files_b)

    # ------------------------------------------
    # 1. PROCESS & PLOT AGGREGATE DATA
    # ------------------------------------------
    print("\nGenerating aggregate plots...")
    df_agg_a = aggregate_all_regions(df_region_a)
    df_agg_b = aggregate_all_regions(df_region_b)
    
    agg_key_cols = ["TRH_BIN", "LENGTH_BIN", "MIN_TAR_GP", "LABEL_TYPE", "AF_BIN_INDEX"]
    merged_agg = pd.merge(df_agg_a, df_agg_b, on=agg_key_cols, suffixes=("_A", "_B"))
    
    merged_agg["R2_ABS_DIFF"] = merged_agg["R2_DS_A"] - merged_agg["R2_DS_B"]
    merged_agg["COUNT_B_SAFE"] = np.where(merged_agg["AF_BIN_COUNT_B"] == 0, np.nan, merged_agg["AF_BIN_COUNT_B"])
    merged_agg["COUNT_REL_DIFF"] = (merged_agg["AF_BIN_COUNT_A"] - merged_agg["AF_BIN_COUNT_B"]) / merged_agg["COUNT_B_SAFE"]
    merged_agg["AF_BIN_MEAN"] = (merged_agg["AF_BIN_MEAN_A"] + merged_agg["AF_BIN_MEAN_B"]) / 2
    
    plot_differences(
        merged_agg, 
        f"Difference: {LABEL_RUN_A} - {LABEL_RUN_B}", 
        f"{OUTPUT_PREFIX}.aggregate"
    )

    # ------------------------------------------
    # 2. PROCESS & PLOT PER-CHROMOSOME DATA
    # ------------------------------------------
    print("\nGenerating per-chromosome plots...")
    reg_key_cols = ["REGION", "TRH_BIN", "LENGTH_BIN", "MIN_TAR_GP", "LABEL_TYPE", "AF_BIN_INDEX"]
    merged_region = pd.merge(df_region_a, df_region_b, on=reg_key_cols, suffixes=("_A", "_B"))
    
    merged_region["R2_ABS_DIFF"] = merged_region["R2_DS_A"] - merged_region["R2_DS_B"]
    merged_region["COUNT_B_SAFE"] = np.where(merged_region["AF_BIN_COUNT_B"] == 0, np.nan, merged_region["AF_BIN_COUNT_B"])
    merged_region["COUNT_REL_DIFF"] = (merged_region["AF_BIN_COUNT_A"] - merged_region["AF_BIN_COUNT_B"]) / merged_region["COUNT_B_SAFE"]
    merged_region["AF_BIN_MEAN"] = (merged_region["AF_BIN_MEAN_A"] + merged_region["AF_BIN_MEAN_B"]) / 2

    # Plot separately for each distinct region found in the merged data
    regions = merged_region["REGION"].unique()
    for region in regions:
        region_df = merged_region[merged_region["REGION"] == region]
        plot_differences(
            region_df, 
            f"Difference: {LABEL_RUN_A} - {LABEL_RUN_B} (Region: {region})", 
            f"{OUTPUT_PREFIX}.{region}"
        )

if __name__ == "__main__":
    main()